In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [3]:
fecha_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=3", con=connection2)
fecha_ini= fecha_ini.iloc[0, 0]

fecha_fin = pd.read_sql_query("SELECT fec_ter FROM etl_act where id_mod=3", con=connection2)
fecha_fin= fecha_fin.iloc[0, 0]

In [4]:
dias_por_intervalo = 15

# Inicializa la fecha actual
fecha_actual = fecha_ini

while fecha_actual <= fecha_fin:
	inicioTiempo = time.time()
	now_inicio = datetime.now()

	fecha_ini_intervalo = fecha_actual
	fecha_fin_intervalo = fecha_actual + timedelta(days=dias_por_intervalo - 1)

	if fecha_fin_intervalo > fecha_fin:
		fecha_fin_intervalo = fecha_fin

	fecha_ini_str = fecha_ini_intervalo.strftime('%Y-%m-%d')
	fecha_fin_str = (fecha_fin_intervalo + timedelta(days=1)).strftime('%Y-%m-%d')
	fecha_fin_display_str = fecha_fin_intervalo.strftime('%Y-%m-%d')

	print(f"Procesando de {fecha_ini_str} al {fecha_fin_display_str}")

	now1 = datetime.now()
	now2 = datetime.now().strftime('%Y-%m-%d')

	query=f"UPDATE etl_act SET fec_act ='{now1}' WHERE id_mod=3"

	c1= text(query)
	connection2.execute(c1)

	tabla='ctppe10'
	col_tabla = "properfec"
	dat= "prog00_essi"
	col_dat='fec_pro'


	oracledb.init_oracle_client()
	oracledb.version = "8.3.0"
	sys.modules["cx_Oracle"] = oracledb
	un = config("USER4_BDI_POSTGRES")
	pw = config("PASS4_BDI_POSTGRES")
	hostname=config("HOST4_BDI_POSTGRES")
	service_name="WNET"
	port = 1527

	engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
			"host": hostname,
			"port": port,
			"service_name": service_name
		}
	)

	connection0 = engine0.connect()


	query1 = f"""
	SELECT
		c.ORICENASICOD,
		c.CENASICOD,
		c.AREHOSCOD,
		c.SERVHOSCOD,
		c.ACTCOD,
		c.ACTESPCOD,
		c.TIPDOCIDENPERCOD,
		c.PERASISDOCIDENNUM,
		c.PROPERFEC,
		c.PROPERTURHORINI,
		c.PROPERTURHORFIN,
		c.PROPERTIPOPROGPERSCOD,
		c.TIPOHORPROGCOD,
		c.PROPERPROHORTOT,
		c.PROPERPROHORMAXNOR,
		c.PROPERPROHORMAXEXT,
		c.ESTPROGCITCOD,
		c.MOTSUSPROGCOD,
		c.PROPERESTREGCOD,
		c.PROPERCREFEC,
		c.PROPERMODFEC,
		c.PROPERSUSFEC,
		c.PROPERANUSUSFEC,
		c.PROPERTURAPEFEC,
		c.PROPERTURCIEFEC,
		c.PROPERTIPOHORDET
	FROM {tabla} c
	WHERE c.{col_tabla} >= TO_DATE('{fecha_ini_str}', 'YYYY-MM-DD') AND c.{col_tabla} < TO_DATE('{fecha_fin_str}', 'YYYY-MM-DD')
	"""

	base1 = pd.read_sql_query(query1, con=connection0)
	connection0.close()
	engine0.dispose()

	base1 = base1.rename(columns={
		'oricenasicod': 'cod_ori',
		'cenasicod': 'cod_cas',
		'arehoscod': 'cod_are',
		'servhoscod': 'cod_ser',
		'actcod': 'cod_act',
		'actespcod': 'cod_sub',
		'tipdocidenpercod': 'cod_tdo',
		'perasisdocidennum': 'num_doc_med',
		'properfec': 'fec_pro',
		'properturhorini': 'hor_tur_ini',
		'properturhorfin': 'hor_tur_fin',
		'propertipoprogperscod': 'cod_tipoprog',
		'tipohorprogcod': 'cod_tiphorprog',
		'properprohortot': 'hor_tot',
		'properprohormaxnor': 'hor_tot_nor',
		'properprohormaxext': 'hor_max_ext',
		'estprogcitcod': 'cod_estprog',
		'motsusprogcod': 'cod_motsus',
		'properestregcod': 'est_reg',
		'propercrefec': 'fec_cre',
		'propermodfec': 'fec_mod',
		'propersusfec': 'fec_sus',
		'properanususfec': 'fec_anu',
		'properturapefec': 'fec_tur_ape',
		'properturciefec': 'fec_tur_cier',
		'propertipohordet': 'tipo_hor_det'
	})

	base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

	oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
	oricas['ori_cod'] = oricas['ori_cod'].str.strip()
	base1['cod_ori'] = base1['cod_ori'].str.strip()
	oricas=oricas.rename(columns={"ori_cod":"cod_ori"})
	base1=pd.merge(base1,oricas,how='left',on='cod_ori')

	base1=base1.drop('cod_ori',axis=1)
	
	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	base1['cod_cas'] = base1['cod_cas'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_cas')

	base1=base1.drop('cod_cas',axis=1)
	
	red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
	red['cod_red'] = red['cod_red'].str.strip()
	base1['cod_red'] = base1['cod_red'].str.strip()
	base1=pd.merge(base1,red,how='left',on='cod_red')

	base1=base1.drop('cod_red',axis=1)

	are = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
	are['cod_are'] = are['cod_are'].str.strip()
	base1['cod_are'] = base1['cod_are'].str.strip()
	base1=pd.merge(base1,are,how='left',on='cod_are')

	base1=base1.drop('cod_are',axis=1)

	serv= pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
	serv['cod_ser'] = serv['cod_ser'].str.strip()
	base1['cod_ser'] = base1['cod_ser'].str.strip()
	base1=pd.merge(base1,serv,how='left',on='cod_ser')

	base1=base1.drop('cod_ser',axis=1)
	
	activi = pd.read_sql_query(f"SELECT id_activi,cod_act FROM dim_activi", con=connection2)
	activi['cod_act'] = activi['cod_act'].str.strip()
	base1['cod_act'] = base1['cod_act'].str.strip()
	activi=activi.rename(columns={"id_activi":"id_acti"})
	base1=pd.merge(base1,activi,how='left',on='cod_act')
	
	subacti = pd.read_sql_query(f"SELECT id_subacti,cod_sub,cod_act FROM dim_subacti", con=connection2)
	subacti['cod_act'] = subacti['cod_act'].str.strip()
	subacti['cod_sub'] = subacti['cod_sub'].str.strip()
	base1['cod_act'] = base1['cod_act'].str.strip()
	base1['cod_sub'] = base1['cod_sub'].str.strip()
	subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
	subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
	base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
	base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
	subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
	base1 = pd.merge(base1,subacti,how='left',on='KEY')

	base1=base1.drop('KEY', axis=1)
	base1=base1.drop('cod_act',axis=1)
	base1=base1.drop('cod_sub',axis=1)

	tipdoc = pd.read_sql_query(f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc", con=connection2)
	tipdoc['cod_tdo'] = tipdoc['cod_tdo'].str.strip()
	base1['cod_tdo'] = base1['cod_tdo'].str.strip()
	tipdoc=tipdoc.rename(columns={"id_tipdoc":"id_tdi_med"})
	base1=pd.merge(base1,tipdoc,how='left',on='cod_tdo')

	base1=base1.drop('cod_tdo', axis=1)

	tippro = pd.read_sql_query(f"SELECT id_tipoprog,cod_tipprog FROM dim_tipoprog", con=connection2)
	tippro=tippro.rename(columns={"cod_tipprog":"cod_tipoprog"})
	tippro['cod_tipoprog'] = tippro['cod_tipoprog'].str.strip()
	base1['cod_tipoprog'] = base1['cod_tipoprog'].str.strip()
	base1=pd.merge(base1,tippro,how='left',on='cod_tipoprog')

	base1=base1.drop('cod_tipoprog', axis=1)

	tiphorpro = pd.read_sql_query(f"SELECT id_tipohorprog,cod_tiphorprog FROM dim_tipohorprog", con=connection2)
	tiphorpro['cod_tiphorprog'] = tiphorpro['cod_tiphorprog'].str.strip()
	base1['cod_tiphorprog'] = base1['cod_tiphorprog'].str.strip()
	base1=pd.merge(base1,tiphorpro,how='left',on='cod_tiphorprog')

	base1=base1.drop('cod_tiphorprog', axis=1)

	estprogcit = pd.read_sql_query(f"SELECT id_estprogcit,cod_estprogcit FROM dim_estprogcit", con=connection2)
	estprogcit['cod_estprogcit'] = estprogcit['cod_estprogcit'].str.strip()
	estprogcit=estprogcit.rename(columns={"cod_estprogcit":"cod_estprog"})
	base1['cod_estprog'] = base1['cod_estprog'].str.strip()
	base1=pd.merge(base1,estprogcit,how='left',on='cod_estprog')

	base1=base1.drop('cod_estprog', axis=1)

	motsusprog = pd.read_sql_query(f"SELECT id_motsuspro,cod_motsus FROM dim_motsuspro", con=connection2)
	motsusprog['cod_motsus'] = motsusprog['cod_motsus'].str.strip()
	base1['cod_motsus'] = base1['cod_motsus'].str.strip()
	base1=pd.merge(base1,motsusprog,how='left',on='cod_motsus')

	base1=base1.drop('cod_motsus', axis=1)
	
	estreg = pd.read_sql_query(f"SELECT id_estreg,cod_est FROM dim_estreg", con=connection2)
	estreg=estreg.rename(columns={"cod_est":"est_reg"})
	estreg['est_reg'] = estreg['est_reg'].str.strip()
	base1['est_reg'] = base1['est_reg'].str.strip()
	base1=pd.merge(base1,estreg,how='left',on='est_reg')

	base1=base1.drop('est_reg',axis=1)

	df1_columns = set(base1.columns)
	df2_columns = set(base2.columns) 
	different_columns = df2_columns - df1_columns
	different_columns

	borrando = f"DELETE FROM {dat} WHERE {col_dat} >='{fecha_ini_str}' and {col_dat} <'{fecha_fin_str}'"
	borrado = connection2.execute(borrando)

	comunes = set(base1.columns).intersection(set(base2.columns)) 
	base1 = base1[list(comunes)]
	base1.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=500000)

	fecha_actual = fecha_fin_intervalo + timedelta(days=1)

	finproceso = time.time()
	tiempoproceso = finproceso - inicioTiempo
	tiempoproceso = round(tiempoproceso, 3)
	print("Proceso completado satisfactoriamente en " + str(tiempoproceso) + " segundos")

query2=f"UPDATE etl_act SET fec_ini ='{now2}' WHERE id_mod=3"
c2= text(query2)
connection2.execute(c2)


Procesando de 2024-07-08 al 2024-07-22
Proceso completado satisfactoriamente en 257.424 segundos
Procesando de 2024-07-23 al 2024-08-06
Proceso completado satisfactoriamente en 195.739 segundos
Procesando de 2024-08-07 al 2024-08-21
Proceso completado satisfactoriamente en 110.9 segundos
Procesando de 2024-08-22 al 2024-09-05
Proceso completado satisfactoriamente en 94.731 segundos
Procesando de 2024-09-06 al 2024-09-20
Proceso completado satisfactoriamente en 88.374 segundos
Procesando de 2024-09-21 al 2024-10-05
Proceso completado satisfactoriamente en 101.538 segundos
Procesando de 2024-10-06 al 2024-10-20
Proceso completado satisfactoriamente en 76.606 segundos
Procesando de 2024-10-21 al 2024-11-04
Proceso completado satisfactoriamente en 65.29 segundos
Procesando de 2024-11-05 al 2024-11-19
Proceso completado satisfactoriamente en 41.089 segundos
Procesando de 2024-11-20 al 2024-12-04
Proceso completado satisfactoriamente en 32.973 segundos
Procesando de 2024-12-05 al 2024-12-19


In [5]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()